In [ ]:
# 1. Configuracion

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA  = "si7006_t2"
TBL_SILVER       = f"{CATALOG}.{SCHEMA}.orders_silver"
TBL_GOLD_ALERTAS = f"{CATALOG}.{SCHEMA}.gold_alertas_reorder"

INITIAL_STOCK_ESTIMATE = 120
REORDER_POINT = 25
CRITICAL_POINT = 10

print(f"Silver:         {TBL_SILVER}")
print(f"Gold alertas:   {TBL_GOLD_ALERTAS}")
print(f"Stock inicial:  {INITIAL_STOCK_ESTIMATE}")
print(f"Reorder point:  {REORDER_POINT}")
print(f"Critical point: {CRITICAL_POINT}")


In [ ]:
# 2. Lectura batch desde Silver

df_silver = spark.table(TBL_SILVER)
print(f"Filas Silver: {df_silver.count():,}")


In [ ]:
# 3. Heuristica de alertas de reorden por producto

df_gold_alertas = (
    df_silver
        .groupBy("stock_code", "description")
        .agg(
            F.sum("quantity").alias("units_sold_total"),
            F.countDistinct("invoice_no").alias("orders_count"),
            F.round(F.sum("total_amount"), 2).alias("gross_revenue"),
            F.max("event_time").alias("last_event_time"),
            F.countDistinct("country").alias("countries_reached")
        )
        .withColumn(
            "estimated_stock_left",
            F.greatest(F.lit(0), F.lit(INITIAL_STOCK_ESTIMATE) - F.col("units_sold_total"))
        )
        .withColumn(
            "stock_consumed_pct",
            F.round((F.col("units_sold_total") / F.lit(INITIAL_STOCK_ESTIMATE)) * 100, 2)
        )
        .withColumn(
            "alert_level",
            F.when(F.col("estimated_stock_left") <= F.lit(CRITICAL_POINT), F.lit("CRITICO"))
             .when(F.col("estimated_stock_left") <= F.lit(REORDER_POINT), F.lit("REORDER"))
             .otherwise(F.lit("OK"))
        )
        .withColumn(
            "alert_reason",
            F.concat(
                F.lit("Ventas acumuladas="),
                F.col("units_sold_total").cast("string"),
                F.lit(", stock estimado restante="),
                F.col("estimated_stock_left").cast("string")
            )
        )
        .filter(F.col("estimated_stock_left") <= F.lit(REORDER_POINT))
        .orderBy(F.col("estimated_stock_left").asc(), F.col("units_sold_total").desc())
)

df_gold_alertas.show(20, truncate=False)


In [ ]:
# 4. Escritura idempotente en Delta

(
    df_gold_alertas.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TBL_GOLD_ALERTAS)
)

print("Tabla gold_alertas_reorder actualizada correctamente")


In [ ]:
# 5. Validacion rapida

spark.table(TBL_GOLD_ALERTAS).printSchema()

spark.sql(f"""
    SELECT alert_level,
           COUNT(*) AS productos,
           MIN(estimated_stock_left) AS stock_minimo,
           MAX(stock_consumed_pct) AS consumo_max_pct
    FROM {TBL_GOLD_ALERTAS}
    GROUP BY alert_level
    ORDER BY stock_minimo ASC
""").show(truncate=False)

spark.sql(f"SELECT * FROM {TBL_GOLD_ALERTAS} ORDER BY estimated_stock_left ASC, units_sold_total DESC LIMIT 20").show(truncate=False)
